In [ ]:
import os
import sys
from pathlib import Path

# Encontra a raiz do projeto (andando para trás a partir de notebooks/)
raiz_projeto = Path(os.getcwd()).resolve()
if raiz_projeto.name == "notebooks":
    raiz_projeto = raiz_projeto.parent

# Adiciona a raiz ao caminho de busca do Python se já não estiver lá
if str(raiz_projeto) not in sys.path:
    sys.path.append(str(raiz_projeto))

# Agora os imports vão funcionar perfeitamente!
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from src.data_loader import carregar_resume_jd_match, garantir_pastas_projeto

# 1. Carrega os dados locais
df = carregar_resume_jd_match()

# 2. Binariza as labels
mapeamento = {'No Fit': 0, 'Potential Fit': 1, 'Good Fit': 1}
df['label_binario'] = df['label'].map(mapeamento)

# 3. Separa os splits
df_train = df[df['split'] == 'train']
df_test = df[df['split'] == 'test']

X_train, y_train = df_train['text'], df_train['label_binario']
X_test, y_test = df_test['text'], df_test['label_binario']

# 4. Vetorização
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 5. Treinamento do Baseline
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_tfidf, y_train)
print("[SUCESSO] Modelo baseline treinado com sucesso!")

ModuleNotFoundError: No module named 'src'

In [ ]:
# 6. Avaliação
y_pred = model.predict(X_test_tfidf)

print("=== METRICAS DO MODELO BASELINE ===")
print("Acurácia:", round(accuracy_score(y_test, y_pred), 4))
print("\nRelatório de Classificação:\n", classification_report(y_test, y_pred, target_names=['No Fit', 'Fit']))
print("\nMatriz de Confusão:\n", confusion_matrix(y_test, y_pred))

# 7. Salva os artefatos locais
pastas = garantir_pastas_projeto()
joblib.dump(model, pastas["models"] / "logistic_regression_baseline.pkl")
joblib.dump(vectorizer, pastas["models"] / "tfidf_vectorizer.pkl")